# 栈帧与调用栈

## 栈帧（stack frame）
- 每发生一次函数调用，Python 都需要保存这一次调用自己的运行状态；承载这些状态的运行时记录称为栈帧
- 同一个函数被调用多次，会产生彼此独立的栈帧，而不是共用一份局部状态
- 一个栈帧中可观察到的核心信息包括：
    1. 正在执行哪段代码
    2. 本次调用的参数和局部名字绑定
    3. 当前执行位置
    4. 调用者对应的上一层栈帧
- 函数的局部命名空间属于这一次调用的栈帧；函数对象是可重复调用的对象，两者不是同一个东西

## 调用栈（call stack）
- 调用栈按照调用关系组织当前仍未结束的栈帧
- 新函数被调用时，新栈帧进入栈顶；函数返回时，它的栈帧退出，控制权回到调用者暂停的位置
- 这种顺序是后进先出：最后进入的函数最先返回
- 在 Notebook 中，代码单元的顶层代码可以理解为 `<module>` 栈帧；它调用的函数依次位于更靠近栈顶的位置

In [1]:
def format_receipt(total):
    print(f'进入 format_receipt: total={total}')
    receipt = f'应付金额: {total:.2f}'
    print('离开 format_receipt')
    return receipt

def calculate_total(price,count):
    print(f'进入 calculate_total: price={price},count={count}')
    total = price*count
    receipt = format_receipt(total)
    print('离开 calculate_total')
    return receipt

result = calculate_total(12.5,4)
print(result)

进入 calculate_total: price=12.5,count=4
进入 format_receipt: total=50.0
离开 format_receipt
离开 calculate_total
应付金额: 50.00


### 上面代码的执行路径
1. `<module>` 调用 `calculate_total(12.5,4)`，创建 `calculate_total` 的栈帧
    - 参数绑定为 `price -> 12.5`、`count -> 4`
    - 局部名字 `total` 绑定到 `50.0`
2. `calculate_total` 调用 `format_receipt(total)`，创建 `format_receipt` 的栈帧
    - `calculate_total` 此时没有结束，只是暂停并等待返回值
    - 参数 `total` 属于 `format_receipt` 这一次调用，与上一层的局部名字相互独立
3. `format_receipt` 返回字符串，它的栈帧退出
4. `calculate_total` 从暂停处继续，把返回值绑定给局部名字 `receipt`，随后返回该字符串并退出
5. `<module>` 得到返回值，将它绑定给全局名字 `result`

调用最深处的活动栈可简化为：

`<module> -> calculate_total -> format_receipt`

箭头表示调用方向，最右侧是当前正在执行的栈顶栈帧。

## 直接观察当前调用栈
- 在当前 CPython 运行时中，`sys._getframe()` 可以取得当前栈帧对象
- `frame.f_code.co_name` 是该栈帧正在执行的代码名称
- `frame.f_back` 指向调用者的栈帧；沿着它向后访问，就能观察当前调用链
- `_getframe()` 以下划线开头，主要适合调试和理解运行机制，不应作为普通业务逻辑的必要依赖

In [2]:
import sys

def show_stack():
    frame = sys._getframe()
    frame_names = []

    while frame is not None:
        name = frame.f_code.co_name
        frame_names.append(name)

        if name == '<module>':
            break

        frame = frame.f_back

    return frame_names

def inner():
    return show_stack()

def outer():
    return inner()

stack_names = outer()
print(' -> '.join(stack_names))

show_stack -> inner -> outer -> <module>


- 输出从当前栈顶 `show_stack` 开始，沿 `f_back` 逐层走向调用者，顺序与调用方向相反
- `show_stack` 返回后，其栈帧先退出；随后 `inner`、`outer` 的栈帧依次退出
- 代码缩进表示函数体、循环体和条件分支的语法层级；调用栈表示运行时尚未结束的调用层级，二者不是同一种结构

## 异常与调用栈
- 如果函数正常 `return`，当前栈帧退出，返回值交给调用者
- 如果函数引发异常且当前层没有处理，异常会沿调用链向调用者传播，同时逐层退出栈帧，这称为栈展开
- traceback 记录异常传播经过的调用位置，因此可以从最外层调用一路定位到最先失败的操作

In [3]:
import traceback

def level_three():
    return 10/0

def level_two():
    return level_three()

def level_one():
    return level_two()

try:
    level_one()
except ZeroDivisionError as error:
    calls = traceback.extract_tb(error.__traceback__)
    call_names = [call.name for call in calls]
    print(f'{type(error).__name__}: {error}')
    print(f"调用路径: {' -> '.join(call_names)}")

ZeroDivisionError: division by zero
调用路径: <module> -> level_one -> level_two -> level_three


### 边界
- 栈帧解释的是一次调用的运行状态，作用域解释的是某处代码查找名字时遵循的规则
- 局部变量通常随栈帧退出而失去常规访问路径，但被闭包或其他对象继续引用的对象仍然可以存活
- Python 层看到的 frame 对象和解释器底层实现使用的机器调用栈不是完全相同的概念；学习 Python 函数调用时，先使用这里的 Python 栈帧模型即可
- 递归调用会让同一个函数产生多个独立栈帧；递归过深会超过解释器允许的递归深度并引发 `RecursionError`